<a href="https://colab.research.google.com/github/saraswathi139/Machine-Learning-Lab/blob/main/ML_EXP_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np

np.random.seed(42)
n = 100

study_hours = np.round(np.random.uniform(1, 10, n), 1)
attendance = np.round(np.random.uniform(50, 100, n), 1)
previous_score = np.round(np.random.uniform(40, 100, n), 1)
internet_access = np.random.choice(["Yes", "No"], n, p=[0.7, 0.3])
extracurricular = np.random.choice(["Yes", "No"], n, p=[0.5, 0.5])

# FinalGrade = weighted formula + noise, so RandomForest has a real relationship to learn
final_grade = (
    (study_hours * 3.5)
    + (attendance * 0.25)
    + (previous_score * 0.35)
    + (np.where(internet_access == "Yes", 5, 0))
    + (np.where(extracurricular == "Yes", 2, 0))
    + np.random.normal(0, 4, n)
)
final_grade = np.clip(final_grade, 0, 100).round(2)

df = pd.DataFrame({
    "Study_Hours": study_hours,
    "Attendance_": attendance,
    "Previous_Score": previous_score,
    "Internet_Access": internet_access,
    "Extracurricular": extracurricular,
    "FinalGrade": final_grade
})

df.to_csv("student_ml_dataset_100_records.csv", index=False)
print("Dataset created: student_ml_dataset_100_records.csv")
df.head()


# ============================================================
# CELL 3 — train_model.py logic (train and save model.pkl)
# ============================================================
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("student_ml_dataset_100_records.csv")

df["Internet_Access"] = df["Internet_Access"].map({"Yes": 1, "No": 0})
df["Extracurricular"] = df["Extracurricular"].map({"Yes": 1, "No": 0})

df.loc[df["Study_Hours"] < 0, "Study_Hours"] = df["Study_Hours"].median()
df.loc[df["Attendance_"] > 100, "Attendance_"] = df["Attendance_"].median()

X = df[["Study_Hours", "Attendance_", "Previous_Score", "Internet_Access", "Extracurricular"]]
y = df["FinalGrade"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model Training Completed")
print("Mean Squared Error:", mse)
print("R2 Score:", r2)

joblib.dump(model, "model.pkl")
print("Model saved as model.pkl")


# ============================================================
# CELL 4 — app.py logic (Flask API, run inside a background thread)
# ============================================================
from flask import Flask, request, jsonify
import joblib
import numpy as np
import threading

app = Flask(__name__)
model = joblib.load("model.pkl")

@app.route("/")
def home():
    return "Student Performance Prediction API is Running!"

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()

    study_hours = data["study_hours"]
    attendance = data["attendance"]
    previous_score = data["previous_score"]
    internet_access = data["internet_access"]
    extracurricular = data["extracurricular"]

    internet_access = 1 if internet_access == "Yes" else 0
    extracurricular = 1 if extracurricular == "Yes" else 0

    input_data = np.array([[study_hours, attendance, previous_score, internet_access, extracurricular]])
    prediction = model.predict(input_data)

    return jsonify({"predicted_final_grade": round(float(prediction[0]), 2)})

def run_app():
    app.run(host="0.0.0.0", port=5000)

# Run Flask in a background thread so the Colab cell doesn't block
thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

print("Flask app started in background on port 5000")


# ============================================================
# CELL 5 — Test the API from inside the same Colab notebook
# ============================================================
import time
time.sleep(2)  # give Flask a moment to start

import requests

# Test home route
response = requests.get("http://127.0.0.1:5000/")
print("Home route response:", response.text)

# Test predict route
payload = {
    "study_hours": 8,
    "attendance": 90,
    "previous_score": 85,
    "internet_access": "Yes",
    "extracurricular": "Yes"
}

response = requests.post("http://127.0.0.1:5000/predict", json=payload)
print("Prediction response:", response.json())

Dataset created: student_ml_dataset_100_records.csv
Model Training Completed
Mean Squared Error: 21.126356256000037
R2 Score: 0.9183018460125417
Model saved as model.pkl
Flask app started in background on port 5000
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [06/Sep/2026 16:11:47] "GET / HTTP/1.1" 200 -
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [06/Sep/2026 16:11:47] "POST /predict HTTP/1.1" 200 -


Home route response: Student Performance Prediction API is Running!
Prediction response: {'predicted_final_grade': 84.69}
